# 01 — Exploração da API de Dados Abertos da Câmara

> "Curiosidade primeiro, código depois." Este notebook responde, com dados reais, às perguntas da Etapa 1 do plano:
> quais endpoints existem, como paginam, que campos trazem, qual a chave natural de cada um e o que sobe para a Silver.

Fontes: `config/openapi_camara.json` (especificação oficial), `config/endpoints.yaml` (catálogo) e as respostas reais
gravadas em `tests/fixtures/api/` por `python -m radar catalog amostras`. Nada aqui chama a API: o notebook roda offline.

In [ ]:
import json
from collections import Counter

import pandas as pd

from radar.catalog import sync
from radar.catalog.amostras import FIXTURES
from radar.catalog.dicionario import perfil_campos
from radar.catalog.loader import load_catalog, topological_levels

pd.set_option("display.max_colwidth", 80)
openapi = sync.ler_openapi()
specs = load_catalog()
fixtures = {
    p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted(FIXTURES.glob("*.json"))
}
n_api = len(sync.paths_get(openapi))
print(f"GET no OpenAPI: {n_api} · catálogo: {len(specs)} · fixtures: {len(fixtures)}")

## 1. Cobertura: o catálogo contém todos os endpoints do OpenAPI?

In [ ]:
cob = sync.cobertura(openapi, specs)
print("faltando:", sorted(cob.faltando) or "nenhum")
print("sobrando:", sorted(cob.sobrando) or "nenhum")
pd.Series(Counter(s.grupo for s in specs), name="endpoints").sort_values(ascending=False).to_frame()

## 2. Tipos, frequência e ordem de extração

In [ ]:
catalogo = pd.DataFrame(
    [
        {
            "endpoint": s.nome,
            "grupo": s.grupo,
            "tipo": s.tipo,
            "paginado": s.paginado,
            "frequencia": s.frequencia,
            "janela": s.janela.tipo,
            "ids_de": ", ".join(s.ids.de) if s.ids else "",
        }
        for s in specs
    ]
)
pd.crosstab(catalogo["tipo"], catalogo["frequencia"], margins=True)

In [ ]:
for i, nivel in enumerate(topological_levels(specs)):
    print(f"nível {i}: {len(nivel)} endpoints — ex.: {', '.join(s.nome for s in nivel[:5])}")

## 3. Envelope e paginação

Toda resposta tem `dados` (lista ou objeto) e `links` (`self`, e `next`/`last` quando há mais páginas).

In [ ]:
envelope = pd.DataFrame(
    [
        {
            "endpoint": nome,
            "formato_dados": fx["_meta"]["tipo_dados"],
            "registros_na_resposta": fx["_meta"]["registros_originais"],
            "links": ", ".join(sorted({lk.get("rel") for lk in fx.get("links", [])})),
        }
        for nome, fx in fixtures.items()
    ]
)
envelope.groupby(["formato_dados", "links"]).size().rename("endpoints").to_frame()

In [ ]:
# Não paginados que devolvem listas grandes de uma vez (atenção à memória no M2)
envelope.query("registros_na_resposta >= 100 and not links.str.contains('next')").sort_values(
    "registros_na_resposta", ascending=False
)

**Achados confirmados nas respostas reais (itens ⚠️ da SPEC):**
- As 4 referências agregadas (`/referencias/deputados`, `/eventos`, `/orgaos`, `/proposicoes`) devolvem `dados` como **objeto de listas** → 1 linha na Bronze, explodida na Silver.
- `/grupos/{id}/membros` devolve o grupo com a lista `membros[]` dentro → chave `{id_pai}, id`.
- `/votacoes/{id}/votos`, `/deputados/{id}/frentes` e `/frentes/{id}/membros` **não paginam**: centenas de registros numa resposta.
- `/votacoes` recusa intervalos **maiores que 3 meses** (HTTP 400), além de exigir datas no mesmo ano.
- Votos e orientações só existem em **votações nominais**; a maioria das votações (comissões, simbólicas) volta vazia.

## 4. Perfil de campos (% de nulos na amostra)

In [ ]:
linhas = []
for nome, fx in fixtures.items():
    for campo, tipo, nulos, total, exemplo in perfil_campos(fx["dados"]):
        linhas.append(
            {
                "endpoint": nome,
                "campo": campo,
                "tipo": tipo,
                "pct_nulos": round(100 * nulos / total),
                "exemplo": exemplo,
            }
        )
campos = pd.DataFrame(linhas)
print(f"{len(campos)} campos observados em {campos['endpoint'].nunique()} endpoints")
campos.groupby("endpoint")["campo"].count().sort_values(ascending=False).head(10).rename(
    "campos"
).to_frame()

In [ ]:
# Sempre nulos na amostra: candidatos a não subir para a Silver (revisar no M2)
campos.query("pct_nulos == 100")[["endpoint", "campo"]].head(30)

## 5. Chaves naturais: os campos da chave existem nos payloads?

In [ ]:
from radar.catalog.amostras import _extrai

verif = []
for s in specs:
    if not isinstance(s.chave, list):
        verif.append({"endpoint": s.nome, "chave": s.chave, "ok": True})
        continue
    dados = fixtures[s.nome]["dados"]
    regs = dados if isinstance(dados, list) else [dados]
    campos_chave = [c for c in s.chave if not c.startswith("{")]
    ok = all(any(_extrai(r, c) for r in regs) for c in campos_chave)
    verif.append({"endpoint": s.nome, "chave": ", ".join(s.chave), "ok": ok})
verif = pd.DataFrame(verif)
print("chaves confirmadas:", verif["ok"].sum(), "de", len(verif))
verif[~verif["ok"]]

## 6. Volume por dia (ordem de grandeza)

In [ ]:
volume = envelope.set_index("endpoint").loc[
    ["eventos", "proposicoes", "votacoes", "deputados", "orgaos", "frentes", "partidos"],
    ["registros_na_resposta"],
]
volume.assign(
    observacao=[
        "eventos na janela (página cheia = 100)",
        "proposições apresentadas na janela (1ª página)",
        "votações na janela (1ª página)",
        "deputados da legislatura (1ª página)",
        "órgãos desde o início da legislatura (1ª página)",
        "frentes da legislatura (1ª página)",
        "partidos da legislatura",
    ]
)

## 7. Hipóteses de negócio (a validar nos marcos M3–M5)

1. **Proposições por tema e semana:** quais temas críticos (Tributário, Tecnologia, Economia) crescem mais?
2. **Tempo de tramitação por tema:** mediana de dias entre apresentação e última tramitação.
3. **Fidelidade partidária e governismo:** votos × orientações (`votacoes_orientacoes`, `votacoes_votos`).
4. **Deputados mais ativos:** autorias, presença em eventos e votações, discursos.
5. **Cota parlamentar:** gasto por deputado, partido, UF, tipo e fornecedor; estornos (valores negativos) em quarentena.
6. **Agenda crítica:** pautas dos próximos 7 dias com proposições de temas críticos (`eventos_pauta`).
7. **Articulação:** frentes, blocos e lideranças ligados a cada tema.

## 8. Decisões para a Silver

- Todas as 78 respostas vão integrais para a Bronze; o mapa Bronze → Silver está em `config/destino_silver.yaml`.
- `cpf` dos deputados fica só na Bronze; nas fixtures, CPFs são mascarados.
- Relações espelhadas (eventos × deputados, órgãos × membros, votações × proposições) são **conformadas** na Silver.
- O dicionário completo, campo a campo, está em `docs/dicionario_de_dados.md` (`python -m radar catalog dicionario`).